# Week 50, Day 3: Comparing models across a group

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w050/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Summed log evidence

Build the 20 by 2 matrix of log evidences: 14 participants favour A by about 3 (sd 0.5), 6 favour B by about 15 (sd 2). Sum each column.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import digamma
r = np.random.default_rng(2009)
lme = np.zeros((20, 2))
lme[:14, 0] = 3 + r.normal(0, 0.5, 14)
lme[14:, 1] = 15 + r.normal(0, 2, 6)
ffx = ...
print(f"fixed effects: log evidence A {ffx[0]:.1f}, B {ffx[1]:.1f}; participants favouring A: {(lme[:, 0] > lme[:, 1]).sum()} of 20")

In [ ]:
import numpy as np
def test_ffx():
    "fixed effects favours model B, although most participants favour A"
    assert ffx[1] > ffx[0] and (lme[:, 0] > lme[:, 1]).sum() > 10

test_ffx()
print("All checks passed")

### Exercise: Variational random-effects BMS

Write rfx_bms(lme, iters=100): start from alpha = alpha0 = ones, and repeat the two updates (subtract each row's maximum of the log u before exponentiating, for numerical stability). Return alpha and g.

In [ ]:
def rfx_bms(lme, iters=100):
    a0 = np.ones(lme.shape[1]); a = a0.copy()
    for _ in range(iters):
        ...
    return a, g

alpha, g = rfx_bms(lme)
freq = alpha / alpha.sum()
print(f"Dirichlet parameters {np.round(alpha, 2)}; expected frequencies A {freq[0]:.3f}, B {freq[1]:.3f}")

In [ ]:
import numpy as np
def test_freq():
    "the expected frequency of model A is between 0.6 and 0.75"
    assert 0.6 < freq[0] < 0.75
def test_assign():
    "each participant is assigned to the model it favours with probability above 0.9"
    assert (g[:14, 0] > 0.9).all() and (g[14:, 1] > 0.9).all()

test_freq()
test_assign()
print("All checks passed")

### Exercise: How sure are we that A is more common?

Draw 100,000 samples from Dirichlet(alpha) and compute the fraction in which model A's frequency exceeds B's. Repeat for a group of 5 made of the first 4 participants and the last.

In [ ]:
def exceedance(alpha, n=100000):
    s = r.dirichlet(alpha, n)
    return ...

xp = exceedance(alpha)
alpha_small, _ = rfx_bms(np.r_[lme[:4], lme[-1:]])
xp_small = exceedance(alpha_small)
print(f"exceedance probability of A: full group {xp:.3f}, group of five {xp_small:.3f}")

In [ ]:
import numpy as np
def test_xp():
    "the exceedance probability of model A is above 0.9"
    assert xp > 0.9
def test_small():
    "with only 5 participants (4 favour A, 1 favours B) it is lower"
    assert xp_small < xp

test_xp()
test_small()
print("All checks passed")